# Pelindungan Privasi Sistem Pengawasan Pintar Berbasis Pendekatan Adversarial Transform pada Cancelable Face Biometrics
**Oleh: Tim Imam Bawang (Universitas Negeri Surabaya)**

Anggota :
- Rizki Piji Fathoni
- Daffa Ahmad Pangreksa
---

## 1. Latar Belakang dan Pernyataan Masalah
Penerapan sistem pengawasan cerdas (*smart surveillance*) masif di ruang publik seringkali berbenturan dengan hak privasi masyarakat. Penggunaan biometrik wajah rentan terhadap ancaman penyalahgunaan data, pencurian identitas, hingga pelanggaran regulasi Pelindungan Data Pribadi (UU PDP No. 27 Tahun 2022). Menyimpan *raw template* (vektor fitur wajah) di dalam *database* pengawasan sama berbahayanya dengan menyimpan kata sandi dalam bentuk *plaintext*. 

Tantangan terbesarnya adalah: **Bagaimana kita dapat melindungi data biometrik tersebut melalui enkripsi yang bisa dibatalkan (*cancelable*), tanpa mengorbankan akurasi/presisi (*utility*) sistem pelacakan CCTV yang umumnya memiliki kualitas citra sangat rendah (*in-the-wild*)?**

## 2. Pipeline Metodologi Notebook
Notebook ini mendemonstrasikan kerangka solusi *end-to-end* untuk masalah di atas melalui kerangka **Cancelable Face Biometrics (CFB)**:
1. **Ekstraksi Fitur Kualitas-Adaptif:** Mengekstrak citra CCTV beresolusi rendah menjadi *embedding* 512 dimensi menggunakan model *state-of-the-art* bersistem *margin-penalty* (AdaFace/ArcFace).
2. **Adversarial Transformation Network:** Mengompresi dan merombak *embedding* mentah menggunakan Jaringan Syaraf Tiruan. Jaringan ini dilatih secara *adversarial* (memanfaatkan *Gradient Reversal Layer*) untuk menghancurkan informasi spasial dan atribut sensitif, namun tetap melestarikan jarak *Euclidean* antar-identitas via *Triplet Loss*.
3. **Orthogonal Permutation (Hashing):** Mengunci vektor hasil dengan matriks ortogonal (Dekomposisi QR) yang dihasilkan dari token pengguna (Kunci Eksternal). Ini menjamin bahwa *template* dapat dibatalkan dan diganti seketika apabila terjadi kebocoran *database*.
4. **Evaluasi Rigid Berstandar Global:** Menguji ketahanan *template* terhadap serangan rekonstruksi mutakhir (*White-box Deep Attacker*), serangan atribut (*Gender/Age*), serta memvalidasi kepatuhan privasi absolut sesuai dengan Standar Internasional ISO/IEC 24745.

## Instalasi

In [ ]:
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install onnxruntime-gpu==1.19.2 nvidia-cudnn-cu12 --quiet
!pip install onnx --quiet
!pip install insightface==0.7.3 --no-deps --quiet
!pip install gdown opencv-python-headless scikit-learn seaborn tqdm scikit-image scipy albumentations PrettyTable --quiet
!find /usr/local/lib/python3.* -type f -name "libcu*.so*" -exec ln -sf {} /usr/lib/x86_64-linux-gnu/ \;
!ln -sf /usr/local/cuda/lib64/libcudart.so.12* /usr/local/cuda/lib64/libcudart.so.13 2>/dev/null || true

import onnxruntime as ort
print("ONNXRuntime providers:", ort.get_available_providers())


## Import

In [ ]:
import os
import cv2
import gdown
import random
import zipfile
import shutil
import gc
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
from scipy.stats import ks_2samp
import onnxruntime as ort
import insightface
from insightface.app import FaceAnalysis


In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
_FIG_COUNTER = [0]

def _fig_slug(fig):
    titles = []
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        titles.append(fig._suptitle.get_text())
    titles += [ax.get_title() for ax in fig.axes if ax.get_title()]
    text = titles[0] if titles else "figure"
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")[:60]
    return slug or "figure"

def _save_open_figures():
    for num in plt.get_fignums():
        fig = plt.figure(num)
        if getattr(fig, "_trace_saved", False) or len(fig.axes) == 0:
            continue
        _FIG_COUNTER[0] += 1
        out_path = os.path.join(FIG_DIR, f"fig{_FIG_COUNTER[0]:02d}_{_fig_slug(fig)}.png")
        fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
        fig._trace_saved = True

if not getattr(plt, "_trace_patched", False):
    _original_show = plt.show
    def _show_and_save(*args, **kwargs):
        _save_open_figures()
        return _original_show(*args, **kwargs)
    plt.show = _show_and_save
    plt._trace_patched = True

    get_ipython().events.callbacks["post_execute"].insert(0, _save_open_figures)

print("Auto-save figure aktif ->", FIG_DIR)


## Konfigurasi
Seed dikunci di 42, 43, dan 44 untuk memastikan hasil eksperimen bisa direproduksi.

In [ ]:
SAMPLED_DATA_DRIVE_ID_5K = "1YGxqs8fOumXdnrldxmOWV2D2qhKvf81i"
SAMPLED_DATA_DRIVE_ID = "1sRGRz0dcQa9n-uHsXIK6O1TN9mufrdKu"

SAMPLED_DATA_DRIVE_ID = SAMPLED_DATA_DRIVE_ID

FULL_DATA_DRIVE_ID = "13ch6BPaexlKt8gXB_I8aX7p1G3yPm2Bl"
USE_FULL_DATA = False
ACTIVE_DRIVE_ID = FULL_DATA_DRIVE_ID if USE_FULL_DATA else SAMPLED_DATA_DRIVE_ID


SEEDS = [42, 43, 44]
RANDOM_SEED = SEEDS[0]
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMBEDDING_DIM = 512
TEMPLATE_DIM = 256
TARGET_FPIR_POINTS = [0.20, 0.10, 0.05, 0.01]
MAX_RANK = 20
N_BOOTSTRAP = 1000
CI_LEVEL = 95
INSTITUTION_KEY_SEED = 20260728
REVOCATION_KEY_SEED = 20260999

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

COLOR_BASELINE_RP = "#1f77b4"
COLOR_BASELINE_BH = "#ff7f0e"
COLOR_PROPOSED = "#2ca02c"
COLOR_PROPOSED_ADAFACE = "#d62728"
COLOR_ACCENT = "#9467bd"
COLOR_NEUTRAL = "#8c564b"

PALETTE_MAIN = [COLOR_BASELINE_RP, COLOR_BASELINE_BH, COLOR_PROPOSED, COLOR_PROPOSED_ADAFACE, COLOR_ACCENT, COLOR_NEUTRAL]
sns.set_palette(PALETTE_MAIN)
print(DEVICE)

## Akuisisi Dataset QMUL SurvFace

In [ ]:
DATASET_ZIP_PATH = "dataset.zip"
EXTRACT_DIR = "qmul_survface_data"
if not os.path.exists(EXTRACT_DIR):
    gdown.download(f"https://drive.google.com/uc?id={ACTIVE_DRIVE_ID}", DATASET_ZIP_PATH, quiet=False)
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
print(os.listdir(EXTRACT_DIR))

## Load & Indexing Data

Untuk mengoptimalkan efisiensi penggunaan memori RAM serta mempercepat waktu eksekusi tanpa mengurangi representativitas eksperimen, dataset yang di-load dibatasi secara terkontrol hingga total **100.000 citra**.

In [ ]:
def find_subset_dir(root, matcher):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        base = os.path.basename(dirpath).lower()
        if matcher(base):
            matches.append(dirpath)
    return matches[0] if matches else None

SUBSET_MATCHERS = {
    "Training_Set": lambda b: "train" in b,
    "Gallery": lambda b: "gallery" in b,
    "Mated_Probe": lambda b: "mated" in b and "unmated" not in b,
    "Unmated_Probe": lambda b: "unmated" in b,
}

SUBSET_PATHS = {name: find_subset_dir(EXTRACT_DIR, matcher) for name, matcher in SUBSET_MATCHERS.items()}
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

def list_images(folder):
    paths = []
    if folder is None or not os.path.exists(folder):
        return paths
    for dirpath, dirnames, filenames in os.walk(folder):
        for f in filenames:
            if f.lower().endswith(IMAGE_EXTENSIONS):
                paths.append(os.path.join(dirpath, f))
    return sorted(paths)

SUBSET_IMAGE_PATHS = {name: list_images(path) for name, path in SUBSET_PATHS.items()}

def parse_identity_camera(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    parts = stem.split("_")
    identity = parts[0]
    camera = parts[1] if len(parts) > 1 else "unknown"
    return identity, camera

records = []
for subset_name, paths in SUBSET_IMAGE_PATHS.items():
    for p in paths:
        identity, camera = parse_identity_camera(p)
        records.append({"path": p, "subset": subset_name, "identity": identity, "camera": camera})
metadata_df = pd.DataFrame(records)

TARGET_QUOTAS = {
    'Gallery': 15000,
    'Mated_Probe': 15000,
    'Unmated_Probe': 20000,
    'Training_Set': 50000
}

gallery_full = metadata_df[metadata_df["subset"] == "Gallery"].reset_index(drop=True)
if len(gallery_full) > TARGET_QUOTAS['Gallery']:
    gallery_df = gallery_full.sample(n=TARGET_QUOTAS['Gallery'], random_state=RANDOM_SEED).reset_index(drop=True)
else:
    gallery_df = gallery_full

valid_gallery_ids = set(gallery_df["identity"])

mated_full = metadata_df[(metadata_df["subset"] == "Mated_Probe") & (metadata_df["identity"].isin(valid_gallery_ids))].reset_index(drop=True)
if len(mated_full) > TARGET_QUOTAS['Mated_Probe']:
    mated_df = mated_full.sample(n=TARGET_QUOTAS['Mated_Probe'], random_state=RANDOM_SEED).reset_index(drop=True)
else:
    mated_df = mated_full

unmated_full = metadata_df[metadata_df["subset"] == "Unmated_Probe"].reset_index(drop=True)
if len(unmated_full) > TARGET_QUOTAS['Unmated_Probe']:
    unmated_df = unmated_full.sample(n=TARGET_QUOTAS['Unmated_Probe'], random_state=RANDOM_SEED).reset_index(drop=True)
else:
    unmated_df = unmated_full

training_full = metadata_df[metadata_df["subset"] == "Training_Set"].reset_index(drop=True)
if len(training_full) > TARGET_QUOTAS['Training_Set']:
    training_df = training_full.sample(n=TARGET_QUOTAS['Training_Set'], random_state=RANDOM_SEED).reset_index(drop=True)
else:
    training_df = training_full

metadata_df = pd.concat([gallery_df, mated_df, unmated_df, training_df], ignore_index=True)

print("Total citra yang berhasil di-index (100k Total):", metadata_df.shape)
metadata_df.head()

## Exploratory Data Analysis (EDA)

### Distribusi Citra dan Identitas Unik per Subset
Melihat seberapa banyak data yang tersedia di tiap subset dan berapa banyak unique person di dalamnya.

In [ ]:
SUBSET_COLOR_MAP = {
    "Training_Set": COLOR_BASELINE_RP,
    "Gallery": COLOR_BASELINE_BH,
    "Mated_Probe": COLOR_PROPOSED,
    "Unmated_Probe": COLOR_ACCENT,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=metadata_df, x="subset", ax=axes[0], palette=SUBSET_COLOR_MAP, hue="subset", legend=False)

for container in axes[0].containers:

    axes[0].bar_label(container, padding=3)
axes[0].set_title("Jumlah Citra per Subset Data")
axes[0].set_xlabel("Subset Data")
axes[0].set_ylabel("Jumlah Citra")

identity_counts = metadata_df.groupby("subset")["identity"].nunique()
bar_colors = [SUBSET_COLOR_MAP.get(s, COLOR_NEUTRAL) for s in identity_counts.index]
bars2 = axes[1].bar(identity_counts.index, identity_counts.values, color=bar_colors)


### Interpretasi: Distribusi Citra dan Subset
Berdasarkan visualisasi di atas, dataset terdistribusi cukup bervariasi antara *Training Set*, *Gallery*, dan *Probe*. Jumlah identitas unik di *Training Set* mendominasi untuk memberikan variasi kelas yang cukup selama masa pelatihan model adversial.

### Distribusi Kamera Pengawas CCTV (Top 15 dan Kategori Lainnya)
Melihat kamera mana yang paling banyak merekam. Hanya Top 15 yang ditampilkan karena jumlah kamera sangat banyak. Perlu dicatat bahwa beberapa kamera mendominasi jauh lebih banyak dari yang lain, yang bisa memengaruhi analisis kualitas.


In [ ]:
camera_counts_all = metadata_df["camera"].value_counts()
median_cam = camera_counts_all.median()
q25_cam = camera_counts_all.quantile(0.25)
q75_cam = camera_counts_all.quantile(0.75)
iqr_cam = q75_cam - q25_cam

top_15_cams = camera_counts_all.nlargest(15)
other_count = camera_counts_all.iloc[15:].sum() if len(camera_counts_all) > 15 else 0
plot_cam_series = pd.concat([top_15_cams, pd.Series({"Lainnya": other_count})]) if other_count > 0 else top_15_cams

plt.figure(figsize=(12, 5))
sns.barplot(x=plot_cam_series.index, y=plot_cam_series.values, color=COLOR_NEUTRAL)
plt.title(f"Distribusi Citra per Kamera (Top 15 + Lainnya | Median: {median_cam:.1f}, IQR: {iqr_cam:.1f})")
plt.xlabel("ID Kamera Pengawas")
plt.ylabel("Jumlah Citra")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Interpretasi: Distribusi Kamera Pengawas
Mayoritas citra hanya ditangkap oleh segelintir kamera dominan, menunjukkan distribusi *long-tail*. Ketidakseimbangan ini menegaskan pentingnya menggunakan metrik CMC dan pengujian lintas-kamera untuk mensimulasikan kondisi pelacakan *cross-camera* di dunia nyata, serta memvalidasi bahwa model tidak semata-mata menghafal *background* kamera.

### Distribusi Ketajaman Dimensi dan Kecerahan Citra CCTV

In [ ]:
def compute_image_stats(path):
    img = cv2.imread(path)
    if img is None:
        return {"width": np.nan, "height": np.nan, "laplacian_var": np.nan, "brightness": np.nan}
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    height, width = gray.shape
    brightness = gray.mean()
    return {"width": width, "height": height, "laplacian_var": laplacian_var, "brightness": brightness}

sample_n = min(1500, len(metadata_df))
quality_sample_df = metadata_df.sample(n=sample_n, random_state=RANDOM_SEED).reset_index(drop=True)
stats_records = [compute_image_stats(p) for p in tqdm(quality_sample_df["path"])]
quality_stats_df = pd.DataFrame(stats_records)
quality_sample_df = pd.concat([quality_sample_df, quality_stats_df], axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.histplot(quality_sample_df["laplacian_var"], bins=40, ax=axes[0], color=COLOR_BASELINE_RP)
axes[0].set_title("Distribusi Ketajaman (Laplacian)")
axes[0].set_xlabel("Varians Laplacian")
sns.histplot(quality_sample_df["width"], bins=40, ax=axes[1], color=COLOR_BASELINE_BH)
axes[1].set_title("Distribusi Lebar Citra Piksel")
axes[1].set_xlabel("Lebar Piksel")
sns.histplot(quality_sample_df["brightness"], bins=40, ax=axes[2], color=COLOR_PROPOSED)
axes[2].set_title("Distribusi Kecerahan Citra")
axes[2].set_xlabel("Rata Rata Intensitas")
plt.tight_layout()
plt.show()

### Interpretasi: Ketajaman Dimensi dan Kecerahan
Terlihat jelas bahwa sebagian besar citra QMUL SurvFace beresolusi sangat rendah (di bawah 64x64 piksel) dengan pencahayaan yang cenderung gelap (Mean RGB rendah). Karakteristik *low-quality* ini mewajibkan kita untuk menggunakan pembobotan *quality-aware* (seperti norma fitur dari AdaFace) agar *backbone* tidak memaksakan *update* gradien pada citra yang terlalu buram.

### Interpretasi: Kurva Pelatihan Adversarial
Grafik menunjukkan konvergensi dari ketiga *loss* utama. *Triplet Loss* menurun stabil, menandakan model berhasil membentuk *clustering* jarak Euclidean antar identitas. Di sisi lain, fluktuasi pada kurva *Adversarial Loss* menunjukkan terjadinya dinamika "tarik-menarik" antara ekstraktor dan diskriminator (Gradient Reversal Layer), memastikan bahwa *embedding* akhir menjadi acak (tidak dapat merekonstruksi wajah) sambil tetap melestarikan relasi kelas.

### Visualisasi Sampel Citra Wajah CCTV QMUL SurvFace

In [ ]:
sample_paths = metadata_df.sample(min(20, len(metadata_df)), random_state=RANDOM_SEED)
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for ax, (_, row) in zip(axes.flatten(), sample_paths.iterrows()):
    if os.path.exists(row["path"]):
        img = Image.open(row["path"])
        ax.imshow(img)
        ax.set_title(f'{row["subset"]}\n{row["identity"]}_{row["camera"]}', fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Interpretasi: Visualisasi Sampel *In-The-Wild*
Sampel citra mengonfirmasi tantangan utama: wajah seringkali terpotong (*occluded*), resolusi sangat pecah, dan orientasi *off-angle*. Transformasi *adversarial* yang kita bangun harus mampu menjaga relasi jarak spasial (*Euclidean*) antar-wajah yang buram ini tanpa membocorkan identitas aslinya.

#

## Ekstraksi Fitur Wajah
Setiap foto wajah diubah menjadi vektor angka 512 dimensi (embedding) menggunakan model ArcFace melalui InsightFace (buffalo_l). Norma vektor fitur sebelum dinormalisasi (feat_norm) turut disimpan sebagai sinyal kualitas citra, sejalan dengan gagasan inti AdaFace bahwa norma fitur berkorelasi dengan kualitas gambar (Kim et al., 2022, CVPR). Sinyal ini dipakai pada tahap berikutnya untuk membangun simulasi AdaFace yang benar-benar adaptif terhadap kualitas (bukan replikasi penuh model dan loss function AdaFace, melainkan pendekatan yang mengikuti prinsip quality-adaptive-nya).


In [ ]:
cuda_options = {"cudnn_conv_algo_search": "DEFAULT", "arena_extend_strategy": "kNextPowerOfTwo"}
face_app = FaceAnalysis(name="buffalo_l", providers=[("CUDAExecutionProvider", cuda_options)])
face_app.prepare(ctx_id=0, det_size=(160, 160))

def extract_face_representation(path):
    img = cv2.imread(path)
    if img is None: return None
    faces = face_app.get(img)
    if len(faces) == 0: return None
    faces = sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]), reverse=True)
    face = faces[0]
    bbox_area = (face.bbox[2] - face.bbox[0]) * (face.bbox[3] - face.bbox[1])
    emb = face.embedding
    feat_norm = float(np.linalg.norm(emb) + 1e-12)
    return emb / feat_norm, feat_norm, int(face.gender), float(face.age), bbox_area

detection_records = []
embedding_records = []
for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
    result = extract_face_representation(row["path"])
    success = result is not None
    detection_records.append({"subset": row["subset"], "camera": row["camera"], "success": success})
    if success:
        arc_emb, feat_norm, gender, age, bbox_area = result
        embedding_records.append({
            "path": row["path"], "subset": row["subset"], "identity": row["identity"],
            "camera": row["camera"], "embedding": arc_emb, "feat_norm": feat_norm,
            "pseudo_gender": gender, "pseudo_age": age, "bbox_area": bbox_area
        })

detection_df = pd.DataFrame(detection_records)
embedding_df = pd.DataFrame(embedding_records)
print("\nTingkat keberhasilan deteksi wajah:", detection_df["success"].mean())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

valid_bboxes = embedding_df[embedding_df['bbox_area'].notnull()]['bbox_area']
if len(valid_bboxes) > 0:
    sns.histplot(np.sqrt(valid_bboxes), bins=40, ax=axes[0], color=COLOR_ACCENT, kde=True)
    axes[0].set_title("Distribusi Ukuran Sisi Bounding Box (Akar Luas)")
    axes[0].set_xlabel("Ukuran Bounding Box (Piksel)")
    axes[0].set_ylabel("Frekuensi")
else:
    axes[0].text(0.5, 0.5, "Tidak ada data Bounding Box valid\n(Semua via Fallback)", ha='center', va='center')
    axes[0].set_axis_off()

camera_success = detection_df.groupby('camera')['success'].agg(['mean', 'count']).reset_index()
camera_success = camera_success[camera_success['count'] > 20]
camera_success = camera_success.sort_values('mean', ascending=False).head(20)

sns.barplot(
    data=camera_success,
    x='camera',
    y='mean',
    ax=axes[1],
    palette=sns.light_palette(COLOR_PROPOSED, n_colors=len(camera_success), reverse=True),
    hue='camera',
    legend=False
)
axes[1].set_title("Tingkat Deteksi Sukses per Kamera (Min. 20 Sampel)")
axes[1].set_xlabel("ID Kamera")
axes[1].set_ylabel("Success Rate (0 - 1)")
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(detection_df["success"].mean(), color=COLOR_PROPOSED_ADAFACE, linestyle='--', label=f'Rata-rata Global: {detection_df["success"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()


### Interpretasi: Ekstraksi Fitur dan Korelasi Kualitas (AdaFace)
Plot norma L2 (*Feature Norm*) menunjukkan korelasi positif dengan luas Bounding Box. Citra dengan resolusi lebih tinggi cenderung memiliki norma fitur yang lebih besar. Hal ini merupakan properti inheren dari *margin-based softmax loss* (AdaFace/ArcFace) yang digunakan untuk menilai seberapa layak (kualitas) sebuah citra diyakini oleh model.

## Penggabungan Embedding dan Penghitungan Identitas Valid
Kalau satu orang punya lebih dari satu foto di gallery, embedding-nya digabung dengan rata-rata berbobot (bobot dari akar luas bounding box wajah), bukan rata-rata polos, supaya foto dengan deteksi wajah yang lebih besar dan lebih jelas memberi kontribusi lebih besar pada representasi akhir. Skor kualitas dan versi AdaFace-simulasi dari tiap embedding juga dihitung di tahap ini, dengan ambang kalibrasi diambil hanya dari Training Set agar tidak terjadi kebocoran informasi dari Gallery maupun Probe. Jumlah identitas yang punya minimal 2 foto di training juga dihitung karena menentukan seberapa banyak data yang bisa dipakai untuk triplet sampling.


In [ ]:
def l2_normalize(vectors):
    array = np.stack(vectors)
    return array / (np.linalg.norm(array, axis=1, keepdims=True) + 1e-12)

def age_to_bucket(age):
    if age < 18:
        return 0
    elif age < 35:
        return 1
    elif age < 50:
        return 2
    return 3

AGE_BUCKET_LABELS = ["Di Bawah 18", "18 Sampai 34", "35 Sampai 49", "50 Ke Atas"]
embedding_df["age_bucket"] = embedding_df["pseudo_age"].apply(age_to_bucket)

gallery_df = embedding_df[embedding_df["subset"] == "Gallery"].reset_index(drop=True)
mated_df = embedding_df[embedding_df["subset"] == "Mated_Probe"].reset_index(drop=True)
unmated_df = embedding_df[embedding_df["subset"] == "Unmated_Probe"].reset_index(drop=True)
training_df = embedding_df[embedding_df["subset"] == "Training_Set"].reset_index(drop=True)

valid_gallery_ids = set(gallery_df["identity"])
mated_df = mated_df[mated_df["identity"].isin(valid_gallery_ids)].reset_index(drop=True)

ADAFACE_ALPHA_MIN = 0.02
ADAFACE_ALPHA_MAX = 0.35

train_feat_norms = training_df["feat_norm"].values
Q_LOW, Q_HIGH = np.percentile(train_feat_norms, [5, 95])
print(f"Batas kualitas (persentil 5/95 feat_norm dari Training Set): {Q_LOW:.4f} / {Q_HIGH:.4f}")

def quality_score_from_norm(feat_norm_array):
    clipped = np.clip(feat_norm_array, Q_LOW, Q_HIGH)
    return (clipped - Q_LOW) / (Q_HIGH - Q_LOW + 1e-12)

def add_adaface_column(df):
    df = df.copy()
    embs = np.stack(df["embedding"].values)
    feat_norms = df["feat_norm"].values
    q = quality_score_from_norm(feat_norms)
    alpha = ADAFACE_ALPHA_MAX - q * (ADAFACE_ALPHA_MAX - ADAFACE_ALPHA_MIN)
    warped = embs * (1.0 + alpha[:, None] * np.tanh(embs))
    warped = warped / (np.linalg.norm(warped, axis=1, keepdims=True) + 1e-12)
    df["embedding_adaface"] = list(warped)
    df["quality_score"] = q
    return df

gallery_df = add_adaface_column(gallery_df)
mated_df = add_adaface_column(mated_df)
unmated_df = add_adaface_column(unmated_df)
training_df = add_adaface_column(training_df)

print(
    "Rentang skor kualitas (0=rendah, 1=tinggi) pada Gallery:",
    f"min={gallery_df['quality_score'].min():.3f}, max={gallery_df['quality_score'].max():.3f}"
)

def quality_weighted_fusion(group, emb_col="embedding"):
    embs = np.stack(group[emb_col].values)
    weights = group["bbox_area"].apply(lambda x: np.sqrt(x) if pd.notnull(x) and x > 0 else 1.0).values
    weights = weights / (np.sum(weights) + 1e-8)
    fused = np.sum(embs * weights[:, None], axis=0)
    return fused / (np.linalg.norm(fused) + 1e-12)

unique_ids = gallery_df['identity'].unique()
gallery_grouped = pd.DataFrame({
    'identity': unique_ids,
    'embedding': [quality_weighted_fusion(gallery_df[gallery_df['identity'] == idx], 'embedding') for idx in unique_ids],
    'embedding_adaface': [quality_weighted_fusion(gallery_df[gallery_df['identity'] == idx], 'embedding_adaface') for idx in unique_ids]
})

gallery_embeddings = np.stack(gallery_grouped["embedding"].values)
gallery_identities_array = gallery_grouped["identity"].to_numpy(dtype=object)
gallery_embeddings_ada = np.stack(gallery_grouped["embedding_adaface"].values)

mated_embeddings = l2_normalize(mated_df["embedding"].tolist())
mated_embeddings_ada = l2_normalize(mated_df["embedding_adaface"].tolist())
unmated_embeddings = l2_normalize(unmated_df["embedding"].tolist())
unmated_embeddings_ada = l2_normalize(unmated_df["embedding_adaface"].tolist())
training_embeddings = l2_normalize(training_df["embedding"].tolist())
training_embeddings_ada = l2_normalize(training_df["embedding_adaface"].tolist())

train_id_counts = training_df["identity"].value_counts()
valid_train_ids = train_id_counts[train_id_counts >= 2].index.tolist()
print(f"Jumlah valid_ids aktual untuk triplet sampling (minimal 2 citra): {len(valid_train_ids)} dari total {training_df['identity'].nunique()} identitas")
print("Dimensi Gallery multi-shot fused:", gallery_embeddings.shape, mated_embeddings.shape, unmated_embeddings.shape)

## Baseline Cancelable: Random Projection dan BioHashing
Dua metode standar (baseline) pelindung template yang dipakai sebagai pembanding. Random Projection memproyeksikan embedding ke dimensi lebih kecil menggunakan matriks kunci rahasia. BioHashing mengubah hasilnya menjadi vektor biner.



In [ ]:
def generate_key_matrix(seed, in_dim, out_dim):
    generator = np.random.default_rng(seed)
    matrix = generator.normal(size=(in_dim, out_dim))
    q, r = np.linalg.qr(matrix) # Gram-Schmidt Orthogonalization untuk BioHashing & RP
    return q

def random_projection_transform(embeddings, key_matrix):
    projected = embeddings @ key_matrix
    norms = np.linalg.norm(projected, axis=1, keepdims=True) + 1e-12
    return projected / norms

def biohash_transform(embeddings, key_matrix):
    projected = embeddings @ key_matrix
    return (projected > 0).astype(np.float32)

def cosine_similarity_matrix(a, b):
    return a @ b.T

def hamming_similarity_matrix(a, b):
    d = a.shape[1]
    matches = (a @ b.T) + ((1.0 - a) @ (1.0 - b).T)
    return matches / d

rp_key_matrix = generate_key_matrix(INSTITUTION_KEY_SEED, EMBEDDING_DIM, TEMPLATE_DIM)
bh_key_matrix = generate_key_matrix(INSTITUTION_KEY_SEED + 1, EMBEDDING_DIM, TEMPLATE_DIM)

gallery_rp = random_projection_transform(gallery_embeddings, rp_key_matrix)
mated_rp = random_projection_transform(mated_embeddings, rp_key_matrix)
unmated_rp = random_projection_transform(unmated_embeddings, rp_key_matrix)
training_rp = random_projection_transform(training_embeddings, rp_key_matrix)

gallery_bh = biohash_transform(gallery_embeddings, bh_key_matrix)
mated_bh = biohash_transform(mated_embeddings, bh_key_matrix)
unmated_bh = biohash_transform(unmated_embeddings, bh_key_matrix)
training_bh = biohash_transform(training_embeddings, bh_key_matrix)

gallery_rp_ada = random_projection_transform(gallery_embeddings_ada, rp_key_matrix)
mated_rp_ada = random_projection_transform(mated_embeddings_ada, rp_key_matrix)
unmated_rp_ada = random_projection_transform(unmated_embeddings_ada, rp_key_matrix)
training_rp_ada = random_projection_transform(training_embeddings_ada, rp_key_matrix)

print("Ukuran template baseline Random Projection dan BioHashing:", gallery_rp.shape, gallery_bh.shape)


#

## Arsitektur Jaringan Transformasi Adversarial
Jaringan ini menerima template dan mengubahnya ke bentuk baru yang tetap bisa dipakai untuk mengenali orang tapi informasi gender dan usia disembunyikan. Ditambahkan pula koneksi residual untuk menjaga representasi identitas, ArcFace Margin Head (Deng et al., 2019) untuk mempertajam pemisahan antar identitas, serta Gradient Reversal Layer (GRL) untuk membalik gradien classifier penyerang.

In [ ]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.lambd = 1.0

    def set_lambda(self, lambd):
        self.lambd = lambd

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambd)


class ProposedTransformNet(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=512, out_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.act = nn.LeakyReLU(0.1)
        self.drop = nn.Dropout(0.2)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)
        self.alpha = nn.Parameter(torch.tensor(0.30))

    def forward(self, x):
        h = self.drop(self.act(self.bn1(self.fc1(x))))
        res = self.bn2(self.fc2(h))
        t = x + torch.clamp(self.alpha, 0.10, 0.50) * res
        return F.normalize(t, p=2, dim=1)


class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=28.0, m=0.25):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = np.cos(m)
        self.sin_m = np.sin(m)
        self.th = np.cos(np.pi - m)
        self.mm = np.sin(np.pi - m) * m

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output


class EnsembleAttributeAdversaryHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.grl = GradientReversalLayer()
        self.head1 = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, n_classes))
        self.head2 = nn.Sequential(nn.Linear(in_dim, 128), nn.LeakyReLU(0.1), nn.Linear(128, n_classes))

    def set_lambda(self, lambd):
        self.grl.set_lambda(lambd)

    def forward(self, x):
        reversed_x = self.grl(x)
        return self.head1(reversed_x), self.head2(reversed_x)


def compute_decorrelation_loss(templates, targets, n_classes=2):
    centered_t = templates - templates.mean(dim=0, keepdim=True)
    if n_classes > 2:
        one_hot = F.one_hot(targets, num_classes=n_classes).float()
        centered_y = one_hot - one_hot.mean(dim=0, keepdim=True)
    else:
        targets_float = (targets.float().unsqueeze(1) if targets.ndim == 1 else targets.float())
        centered_y = targets_float - targets_float.mean(dim=0, keepdim=True)
    cov = torch.matmul(centered_t.T, centered_y) / (templates.size(0) - 1 + 1e-8)
    return torch.mean(cov**2)


def sample_random_triplets(embeddings, identities, batch_size, seed):
    generator = np.random.default_rng(seed)
    id_to_indices = {}
    for idx, identity in enumerate(identities):
        id_to_indices.setdefault(identity, []).append(idx)
    valid_ids = [i for i, idxs in id_to_indices.items() if len(idxs) >= 2]
    anchors, positives, negatives = [], [], []
    selected_ids = generator.choice(valid_ids, size=min(batch_size, len(valid_ids)), replace=False)
    for anchor_id in selected_ids:
        a_idx, p_idx = generator.choice(id_to_indices[anchor_id], size=2, replace=False)
        other_ids = [i for i in valid_ids if i != anchor_id]
        neg_id = generator.choice(other_ids)
        n_idx = generator.choice(id_to_indices[neg_id])
        anchors.append(a_idx)
        positives.append(p_idx)
        negatives.append(n_idx)
    return np.array(anchors), np.array(positives), np.array(negatives)

### Pelatihan Mini-Batch Adversarial
Pelatihan dilakukan per batch kecil (128 sampel) sebanyak 1600 langkah (40 epoch x 40 langkah per epoch) dengan masa warmup 10 epoch awal untuk mengunci identitas sebelum GRL diaktifkan. Jaringan dilatih di atas embedding AdaFace-adaptif (training_rp_ada), domain yang sama dengan yang dipakai saat inferensi pada Gallery dan Probe, sehingga tidak ada mismatch antara domain data latih dan domain data uji. Hasil akhir dikalikan dengan matriks kunci ortogonal (ISO/IEC 24745) agar akurasi identifikasi tetap tinggi sekaligus menjaga templat dari serangan reidentifikasi. Seluruh proses pelatihan diulang untuk 3 seed berbeda guna melaporkan stabilitas Rank-1 Accuracy Metode Usulan (rata-rata dan standar deviasi).


In [ ]:
BATCH_SIZE = 128
N_EPOCHS = 40
WARMUP_EPOCHS = 10
STEPS_PER_EPOCH = 40
TOTAL_STEPS = N_EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = WARMUP_EPOCHS * STEPS_PER_EPOCH
TRIPLET_MARGIN = 0.35
HIDDEN_DIM = 256

training_proposed_input_tensor = torch.tensor(training_rp_ada, dtype=torch.float32).to(DEVICE)
training_gender_tensor = torch.tensor(training_df["pseudo_gender"].values, dtype=torch.long).to(DEVICE)
training_age_tensor = torch.tensor(training_df["age_bucket"].values, dtype=torch.long).to(DEVICE)
training_identity_array = training_df["identity"].to_numpy(dtype=object)

unique_train_ids = np.unique(training_identity_array)
id_to_label_map = {uid: idx for idx, uid in enumerate(unique_train_ids)}
training_id_labels_tensor = torch.tensor([id_to_label_map[uid] for uid in training_identity_array], dtype=torch.long).to(DEVICE)
N_TRAIN_CLASSES = len(unique_train_ids)

raw_perm_matrix = generate_key_matrix(INSTITUTION_KEY_SEED + 99, TEMPLATE_DIM, TEMPLATE_DIM)
perm_key_matrix, _ = np.linalg.qr(raw_perm_matrix)

def train_proposed_pipeline(seed):
    torch.manual_seed(seed)
    net = ProposedTransformNet(TEMPLATE_DIM, HIDDEN_DIM, TEMPLATE_DIM).to(DEVICE)
    arc_head = ArcMarginProduct(TEMPLATE_DIM, N_TRAIN_CLASSES, s=28.0, m=0.25).to(DEVICE)
    gen_adv = EnsembleAttributeAdversaryHead(TEMPLATE_DIM, 2).to(DEVICE)
    age_adv = EnsembleAttributeAdversaryHead(TEMPLATE_DIM, len(AGE_BUCKET_LABELS)).to(DEVICE)

    optimizer_main = torch.optim.AdamW(list(net.parameters()) + list(arc_head.parameters()), lr=4e-4, weight_decay=1e-3)
    optimizer_adv = torch.optim.AdamW(list(gen_adv.parameters()) + list(age_adv.parameters()), lr=2e-4, weight_decay=1e-3)
    scheduler_main = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_main, T_max=TOTAL_STEPS, eta_min=1e-5)

    triplet_loss_fn = nn.TripletMarginLoss(margin=TRIPLET_MARGIN)
    ce_loss_id_fn = nn.CrossEntropyLoss()
    ce_loss_gender_fn = nn.CrossEntropyLoss()
    ce_loss_age_fn = nn.CrossEntropyLoss()

    history = {"triplet_loss": [], "gender_loss": [], "age_loss": [], "decorr_loss": []}
    global_step = 0

    for epoch in range(N_EPOCHS):
        for step in range(STEPS_PER_EPOCH):
            global_step += 1

            if global_step <= WARMUP_STEPS:
                lambd = 0.0
                adv_weight = 0.0
            else:
                p = (global_step - WARMUP_STEPS) / (TOTAL_STEPS - WARMUP_STEPS)
                lambd = float(2.0 / (1.0 + np.exp(-4 * p)) - 1.0)
                adv_weight = 1.0 * lambd

            gen_adv.set_lambda(lambd)
            age_adv.set_lambda(lambd)

            anchors, positives, negatives = sample_random_triplets(training_rp_ada, training_identity_array, BATCH_SIZE, seed + global_step)
            anchor_template = net(training_proposed_input_tensor[anchors])
            positive_template = net(training_proposed_input_tensor[positives])
            negative_template = net(training_proposed_input_tensor[negatives])
            loss_triplet = triplet_loss_fn(anchor_template, positive_template, negative_template)

            perm_idx = torch.randperm(len(training_rp_ada))[:BATCH_SIZE].to(DEVICE)
            batch_template = net(training_proposed_input_tensor[perm_idx])
            batch_id_labels = training_id_labels_tensor[perm_idx]
            g_y = training_gender_tensor[perm_idx]
            a_y = training_age_tensor[perm_idx]

            arc_logits = arc_head(batch_template, batch_id_labels)
            loss_id_arc = ce_loss_id_fn(arc_logits, batch_id_labels)

            g_logits1, g_logits2 = gen_adv(batch_template)
            a_logits1, a_logits2 = age_adv(batch_template)
            loss_gender = 0.5 * (ce_loss_gender_fn(g_logits1, g_y) + ce_loss_gender_fn(g_logits2, g_y))
            loss_age = 0.5 * (ce_loss_age_fn(a_logits1, a_y) + ce_loss_age_fn(a_logits2, a_y))
            loss_decorr = compute_decorrelation_loss(batch_template, g_y, 2) + compute_decorrelation_loss(batch_template, a_y, len(AGE_BUCKET_LABELS))

            total_loss = 1.0 * loss_triplet + 1.0 * loss_id_arc + adv_weight * (1.0 * loss_gender + 2.0 * loss_age + 0.25 * loss_decorr)

            optimizer_main.zero_grad()
            optimizer_adv.zero_grad()
            total_loss.backward()
            optimizer_main.step()
            scheduler_main.step()

            if adv_weight > 0:
                for k in range(2):
                    adv_perm = torch.randperm(len(training_rp_ada))[:BATCH_SIZE].to(DEVICE)
                    with torch.no_grad():
                        adv_batch_template = net(training_proposed_input_tensor[adv_perm]).detach()
                    g_y2 = training_gender_tensor[adv_perm]
                    a_y2 = training_age_tensor[adv_perm]
                    g_l1, g_l2 = gen_adv(adv_batch_template)
                    a_l1, a_l2 = age_adv(adv_batch_template)
                    loss_adv_extra = 0.5 * (ce_loss_gender_fn(g_l1, g_y2) + ce_loss_gender_fn(g_l2, g_y2)) + 0.5 * (ce_loss_age_fn(a_l1, a_y2) + ce_loss_age_fn(a_l2, a_y2))
                    optimizer_adv.zero_grad()
                    loss_adv_extra.backward()
                    optimizer_adv.step()

            history["triplet_loss"].append(loss_triplet.item())
            history["gender_loss"].append(loss_gender.item())
            history["age_loss"].append(loss_age.item())
            history["decorr_loss"].append(loss_decorr.item())

    return net, history

seed_models = {}
seed_histories = {}
for s in SEEDS:
    model_s, hist_s = train_proposed_pipeline(s)
    seed_models[s] = model_s
    seed_histories[s] = hist_s

proposed_net = seed_models[RANDOM_SEED]
history = seed_histories[RANDOM_SEED]
proposed_net.eval()

def apply_cancellable_key(templates, perm_matrix):
    transformed = templates @ perm_matrix
    return transformed / (np.linalg.norm(transformed, axis=1, keepdims=True) + 1e-12)

with torch.no_grad():
    raw_gallery_proposed = proposed_net(torch.tensor(gallery_rp_ada, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    raw_mated_proposed = proposed_net(torch.tensor(mated_rp_ada, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    raw_unmated_proposed = proposed_net(torch.tensor(unmated_rp_ada, dtype=torch.float32).to(DEVICE)).cpu().numpy()

gallery_proposed = apply_cancellable_key(raw_gallery_proposed, perm_key_matrix)
mated_proposed = apply_cancellable_key(raw_mated_proposed, perm_key_matrix)
unmated_proposed = apply_cancellable_key(raw_unmated_proposed, perm_key_matrix)

gc.collect()
torch.cuda.empty_cache()

true_mated_ids_seedcheck = mated_df["identity"].to_numpy(dtype=object)
seed_rank1_scores = []
for s in SEEDS:
    net_s = seed_models[s]
    net_s.eval()
    with torch.no_grad():
        g_s = apply_cancellable_key(
            net_s(torch.tensor(gallery_rp_ada, dtype=torch.float32).to(DEVICE)).cpu().numpy(),
            perm_key_matrix,
        )
        m_s = apply_cancellable_key(
            net_s(torch.tensor(mated_rp_ada, dtype=torch.float32).to(DEVICE)).cpu().numpy(),
            perm_key_matrix,
        )
    sim_s = m_s @ g_s.T
    pred_s = gallery_identities_array[np.argmax(sim_s, axis=1)]
    rank1_s = (pred_s == true_mated_ids_seedcheck).mean()
    seed_rank1_scores.append(rank1_s)
    print(f"Seed {s}: Rank-1 Accuracy Metode Usulan = {rank1_s:.4f}")

print(
    f"Stabilitas lintas-seed -> Rank-1 mean={np.mean(seed_rank1_scores):.4f}, "
    f"std={np.std(seed_rank1_scores):.4f} (n_seed={len(SEEDS)})"
)

### Kurva Pelatihan
Garis merah putus-putus adalah batas acak (chance level): kalau classifier tidak bisa belajar apa-apa, loss-nya akan berada di sekitar garis ini. Loss adversarial yang bertahan di atas garis berarti atribut gender/usia berhasil disembunyikan dari template.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(history["triplet_loss"], color=COLOR_PROPOSED)
axes[0].set_title("Loss Triplet Penjaga Identitas")
axes[0].set_xlabel("Langkah Optimasi Mini-Batch")
axes[0].set_ylabel("Nilai Loss")

axes[1].plot(history["gender_loss"], color=COLOR_ACCENT)
axes[1].axhline(np.log(2.0), color="red", linestyle="--", linewidth=1.5, label="Chance Level ln(2)=0.693")
axes[1].set_title("Loss Adversarial Jenis Kelamin")
axes[1].set_xlabel("Langkah Optimasi Mini-Batch")
axes[1].set_ylabel("Nilai Loss")
axes[1].legend()

axes[2].plot(history["age_loss"], color=COLOR_NEUTRAL)
axes[2].axhline(np.log(4.0), color="red", linestyle="--", linewidth=1.5, label="Chance Level ln(4)=1.386")
axes[2].set_title("Loss Adversarial Kelompok Usia")
axes[2].set_xlabel("Langkah Optimasi Mini-Batch")
axes[2].set_ylabel("Nilai Loss")
axes[2].legend()

plt.tight_layout()
plt.show()

### Interpretasi: Kurva Pelatihan Adversarial
Grafik menunjukkan konvergensi dari ketiga *loss* utama. *Triplet Loss* menurun stabil, menandakan model berhasil membentuk *clustering* jarak Euclidean antar identitas. Di sisi lain, fluktuasi pada kurva *Adversarial Loss* menunjukkan terjadinya dinamika "tarik-menarik" antara ekstraktor dan diskriminator (Gradient Reversal Layer), memastikan bahwa *embedding* akhir menjadi acak (tidak dapat merekonstruksi wajah) sambil tetap melestarikan relasi kelas.

## Basis Data Terlindungi
Hanya template yang disimpan, foto asli tidak. Note: satu kunci dipakai untuk seluruh database.

In [ ]:
protected_database = pd.DataFrame({
    "identity": gallery_identities_array,
    "template_rp": list(gallery_rp),
    "template_bh": list(gallery_bh),
    "template_proposed": list(gallery_proposed),
})
print("Ancamam threat model: Penggunaan kunci tunggal bersama untuk seluruh gallery berisiko kompromi massal jika kunci terkompromi.")
print(protected_database.shape)
protected_database.head()

## Protokol Pencarian 1:N dan Pengungkapan Selektif
Foto pelaku dibandingkan dengan seluruh isi database, lalu diurutkan berdasarkan kemiripan. Ambang pengungkapan dikalibrasi dari distribusi skor Unmated Probe pada target False Positive Identification Rate (FPIR) operasional tertentu, bukan sekadar nilai tengah (median), sehingga tingkat kesalahan pengungkapan identitas dapat dikendalikan sesuai kebutuhan operasional. Hanya identitas yang melewati ambang batas tersebut yang datanya "dibuka", sisanya tetap terkunci. Ini adalah bentuk teknis dari prinsip minimalisasi data di UU PDP No. 27/2022.


In [ ]:
def identification_search(probe_templates, gallery_templates, gallery_identities, similarity_fn):
    similarity = similarity_fn(probe_templates, gallery_templates)
    ranking = np.argsort(-similarity, axis=1)
    ranked_identities = gallery_identities[ranking]
    ranked_scores = np.take_along_axis(similarity, ranking, axis=1)
    return ranked_identities, ranked_scores

def selective_disclosure(ranked_identities, ranked_scores, threshold):
    disclosed = ranked_scores[:, 0] >= threshold
    disclosed_identity = np.where(disclosed, ranked_identities[:, 0], "TERKUNCI")
    return disclosed, disclosed_identity

def calibrate_threshold_for_fpir(unmated_top_scores, target_fpir):
    return float(np.percentile(unmated_top_scores, 100.0 * (1.0 - target_fpir)))

METHODS = {
    "Random Projection": {"gallery": gallery_rp, "mated": mated_rp, "unmated": unmated_rp, "similarity_fn": cosine_similarity_matrix, "color": COLOR_BASELINE_RP},
    "BioHashing": {"gallery": gallery_bh, "mated": mated_bh, "unmated": unmated_bh, "similarity_fn": hamming_similarity_matrix, "color": COLOR_BASELINE_BH},
    "Metode Usulan": {"gallery": gallery_proposed, "mated": mated_proposed, "unmated": unmated_proposed, "similarity_fn": cosine_similarity_matrix, "color": COLOR_PROPOSED},
}

search_results = {}
for method_name, cfg in METHODS.items():
    mated_ranked_ids, mated_ranked_scores = identification_search(cfg["mated"], cfg["gallery"], gallery_identities_array, cfg["similarity_fn"])
    unmated_ranked_ids, unmated_ranked_scores = identification_search(cfg["unmated"], cfg["gallery"], gallery_identities_array, cfg["similarity_fn"])
    search_results[method_name] = {
        "mated_ranked_ids": mated_ranked_ids,
        "mated_ranked_scores": mated_ranked_scores,
        "unmated_ranked_ids": unmated_ranked_ids,
        "unmated_ranked_scores": unmated_ranked_scores,
    }

TARGET_OPERATIONAL_FPIR = 0.01
operational_threshold = calibrate_threshold_for_fpir(
    search_results["Metode Usulan"]["unmated_ranked_scores"][:, 0], TARGET_OPERATIONAL_FPIR
)
disclosed_flags, disclosed_identities = selective_disclosure(
    search_results["Metode Usulan"]["mated_ranked_ids"],
    search_results["Metode Usulan"]["mated_ranked_scores"],
    operational_threshold,
)
print(f"Ambang batas pengungkapan selektif (dikalibrasi pada target FPIR={TARGET_OPERATIONAL_FPIR}):", operational_threshold)
print("Proporsi identitas mated yang terungkap pada ambang ini (perkiraan TPIR operasional):", disclosed_flags.mean())

## Evaluasi Identifikasi 1:N (CMC dan DIR/FPIR)
CMC mengukur seberapa sering target ada di dalam hasil teratas (top-k). DIR/FPIR mengukur seberapa banyak target yang terdeteksi pada berbagai tingkat alarm palsu. Dua grafik ini adalah standar evaluasi sistem identifikasi biometrik open-set (Phillips et al., NIST).

In [ ]:
def compute_cmc(ranked_ids, true_ids, max_rank):
    cmc = np.zeros(max_rank)
    for k in range(1, max_rank + 1):
        hits = (ranked_ids[:, :k] == true_ids[:, None]).any(axis=1)
        cmc[k - 1] = hits.mean()
    return cmc

def compute_tpir_fpir(mated_ranked_ids, mated_ranked_scores, mated_true_ids, unmated_ranked_scores, thresholds):
    rank1_correct = mated_ranked_ids[:, 0] == mated_true_ids
    mated_top_score = mated_ranked_scores[:, 0]
    unmated_top_score = unmated_ranked_scores[:, 0]
    tpir_list, fpir_list = [], []
    for th in thresholds:
        accepted_and_correct = (mated_top_score >= th) & rank1_correct
        tpir_list.append(accepted_and_correct.mean())
        fpir_list.append((unmated_top_score >= th).mean())
    return np.array(tpir_list), np.array(fpir_list)

def tpir_at_fpir(fpir_vals, tpir_vals, target_fpir):
    order = np.argsort(fpir_vals)
    return float(np.interp(target_fpir, fpir_vals[order], tpir_vals[order]))

true_mated_ids = mated_df["identity"].to_numpy(dtype=object)
cmc_curves = {}
dir_curves = {}
for method_name in METHODS:
    ranked_ids = search_results[method_name]["mated_ranked_ids"]
    cmc_curves[method_name] = compute_cmc(ranked_ids, true_mated_ids, MAX_RANK)
    m_scores = search_results[method_name]["mated_ranked_scores"]
    u_scores = search_results[method_name]["unmated_ranked_scores"]
    th_range = np.linspace(min(m_scores.min(), u_scores.min()), max(m_scores.max(), u_scores.max()), 300)
    tpir, fpir = compute_tpir_fpir(ranked_ids, m_scores, true_mated_ids, u_scores, th_range)
    dir_curves[method_name] = {"tpir": tpir, "fpir": fpir}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for method_name, cfg in METHODS.items():
    axes[0].plot(range(1, MAX_RANK + 1), cmc_curves[method_name], label=method_name, color=cfg["color"], marker="o", markersize=3)
axes[0].set_title("Kurva Cumulative Match Characteristic (CMC)")
axes[0].set_xlabel("Peringkat Rank k")
axes[0].set_ylabel("Tingkat Identifikasi Kumulatif")
axes[0].legend()

for method_name, cfg in METHODS.items():
    f_v = dir_curves[method_name]["fpir"]
    t_v = dir_curves[method_name]["tpir"]
    ord_i = np.argsort(f_v)
    axes[1].plot(f_v[ord_i] + 1e-6, t_v[ord_i], label=method_name, color=cfg["color"])
axes[1].set_title("Kurva Detection and Identification Rate (DIR)")
axes[1].set_xlabel("False Positive Identification Rate (FPIR)")
axes[1].set_ylabel("True Positive Identification Rate (TPIR)")
axes[1].set_xscale("log")
axes[1].legend()

plt.tight_layout()
plt.show()

summary_rows = []
for method_name in METHODS:
    row = {"Metode": method_name}
    for target in TARGET_FPIR_POINTS:
        row[f"TPIR pada FPIR {target}"] = tpir_at_fpir(dir_curves[method_name]["fpir"], dir_curves[method_name]["tpir"], target)
    row["Rank 1 Accuracy"] = cmc_curves[method_name][0]
    row["Rank 5 Accuracy"] = cmc_curves[method_name][4]
    summary_rows.append(row)
identification_summary_df = pd.DataFrame(summary_rows)
identification_summary_df

## Interval Kepercayaan Bootstrap
Bootstrap dijalankan dengan cara resampling per identitas (bukan per foto) agar varians yang dihitung akurat. Paired bootstrap dipakai untuk menguji apakah perbedaan performa antar metode betul-betul signifikan atau hanya kebetulan.


In [ ]:
def cluster_block_bootstrap_ci(identities, correct_mask, n_boot=1000, ci_level=95, seed=42):
    generator = np.random.default_rng(seed)
    unique_ids = np.unique(identities)
    n_clusters = len(unique_ids)
    id_to_indices = {uid: np.where(identities == uid)[0] for uid in unique_ids}
    boot_means = np.empty(n_boot)
    for i in range(n_boot):
        boot_ids = generator.choice(unique_ids, size=n_clusters, replace=True)
        boot_indices = np.concatenate([id_to_indices[uid] for uid in boot_ids])
        boot_means[i] = correct_mask[boot_indices].mean()
    low_pct = (100 - ci_level) / 2.0
    high_pct = 100 - low_pct
    return correct_mask.mean(), np.percentile(boot_means, low_pct), np.percentile(boot_means, high_pct)

def paired_bootstrap_diff(identities, mask_a, mask_b, n_boot=1000, ci_level=95, seed=42):
    generator = np.random.default_rng(seed)
    unique_ids = np.unique(identities)
    n_clusters = len(unique_ids)
    id_to_indices = {uid: np.where(identities == uid)[0] for uid in unique_ids}
    diff_means = np.empty(n_boot)
    for i in range(n_boot):
        boot_ids = generator.choice(unique_ids, size=n_clusters, replace=True)
        boot_indices = np.concatenate([id_to_indices[uid] for uid in boot_ids])
        diff_means[i] = mask_a[boot_indices].mean() - mask_b[boot_indices].mean()
    low_pct = (100 - ci_level) / 2.0
    high_pct = 100 - low_pct
    mean_diff = mask_a.mean() - mask_b.mean()
    return mean_diff, np.percentile(diff_means, low_pct), np.percentile(diff_means, high_pct)

rank1_ci_rows = []
for method_name in METHODS:
    r_ids = search_results[method_name]["mated_ranked_ids"]
    c_mask = (r_ids[:, 0] == true_mated_ids).astype(float)
    m_val, low, high = cluster_block_bootstrap_ci(true_mated_ids, c_mask, N_BOOTSTRAP, CI_LEVEL, RANDOM_SEED)
    rank1_ci_rows.append({"Metode": method_name, "Rank 1 Accuracy": m_val, "Batas Bawah": low, "Batas Atas": high})
rank1_ci_df = pd.DataFrame(rank1_ci_rows)

mask_proposed = (search_results["Metode Usulan"]["mated_ranked_ids"][:, 0] == true_mated_ids).astype(float)
mask_rp = (search_results["Random Projection"]["mated_ranked_ids"][:, 0] == true_mated_ids).astype(float)
mean_d, low_d, high_d = paired_bootstrap_diff(true_mated_ids, mask_proposed, mask_rp, N_BOOTSTRAP, CI_LEVEL, RANDOM_SEED)
print(f"Paired Bootstrap Difference (Metode Usulan - RP): {mean_d:.4f} [95% CI: {low_d:.4f}, {high_d:.4f}]")

plt.figure(figsize=(8, 5))
x_pos = np.arange(len(rank1_ci_df))
bars = plt.bar(
    x_pos,
    rank1_ci_df["Rank 1 Accuracy"],
    yerr=[rank1_ci_df["Rank 1 Accuracy"] - rank1_ci_df["Batas Bawah"], rank1_ci_df["Batas Atas"] - rank1_ci_df["Rank 1 Accuracy"]],
    capsize=6,
    color=[METHODS[m]["color"] for m in rank1_ci_df["Metode"]],
)
plt.bar_label(bars, fmt='%.4f', label_type='edge', padding=15)


## Uji Standar ISO/IEC 24745
Standar internasional untuk sistem cancelable biometrics mensyaratkan 4 properti. Dua sudah diuji di bagian lain (irreversibility dan performance). Bagian ini menguji dua sisanya: unlinkability (apakah template dua orang berbeda bisa dibedakan distribusinya) dan revocability (kalau kunci diganti, apakah template lama dan baru benar-benar tidak bisa dicocokkan).


In [ ]:
print("Mengevaluasi Unlinkability dan Revocability sesuai standar ISO/IEC 24745...")

REVOCATION_KEY_SEED = 999
revocation_rp_key = generate_key_matrix(REVOCATION_KEY_SEED, EMBEDDING_DIM, TEMPLATE_DIM)

generator_rev = np.random.default_rng(REVOCATION_KEY_SEED)
rev_perm_key_matrix = generator_rev.normal(size=(TEMPLATE_DIM, TEMPLATE_DIM))
q_rev, _ = np.linalg.qr(rev_perm_key_matrix)
rev_perm_key_matrix = q_rev

rev_gallery_rp = random_projection_transform(gallery_embeddings_ada, revocation_rp_key)
with torch.no_grad():
    rev_gallery_proposed = proposed_net(torch.tensor(rev_gallery_rp, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    rev_gallery_proposed = rev_gallery_proposed @ rev_perm_key_matrix
    rev_gallery_proposed = rev_gallery_proposed / (np.linalg.norm(rev_gallery_proposed, axis=1, keepdims=True) + 1e-12)

gallery_proposed_final = gallery_proposed

cross_sim = cosine_similarity_matrix(gallery_proposed_final, rev_gallery_proposed)

gal_ids_col = gallery_identities_array.reshape(-1, 1)
mated_mask = (gal_ids_col == gal_ids_col.T)
cross_mated_scores = cross_sim[mated_mask].flatten()

unmated_mask = ~mated_mask
cross_unmated_scores = cross_sim[unmated_mask].flatten()
np.random.shuffle(cross_unmated_scores)
if len(cross_mated_scores) > 0:
    cross_unmated_scores = cross_unmated_scores[:len(cross_mated_scores)]

from scipy.stats import ks_2samp
ks_stat, p_val = ks_2samp(cross_mated_scores, cross_unmated_scores)
hist_m, bin_edges = np.histogram(cross_mated_scores, bins=50, density=True)
hist_u, _ = np.histogram(cross_unmated_scores, bins=bin_edges, density=True)
overlap = np.sum(np.minimum(hist_m, hist_u) * np.diff(bin_edges))

print(f"KS Statistic Unlinkability: {ks_stat:.5f}")
print(f"Overlap Coefficient: {overlap:.5f}")

rank_rev = np.argsort(-cross_sim, axis=1)
ranked_ids_rev = gallery_identities_array[rank_rev]
cross_match_rate = (ranked_ids_rev[:, 0] == gallery_identities_array).mean()
print(f"Cross-Match Rate (Revocability): {cross_match_rate:.4f}")

iso_summary_df = pd.DataFrame([{
    "Metode": "Metode Usulan",
    "KS Statistic Unlinkability": ks_stat,
    "Overlap Coefficient": overlap,
    "Revocability Cross-Match Rate": cross_match_rate
}])
iso_summary_df

### Interpretasi: Kepatuhan ISO/IEC 24745 (Unlinkability)
Kurva hijau (*Unmated*) dan merah (*Mated*) saling tumpang tindih nyaris sempurna (*Overlap Coefficient* ~0.937). *KS Statistic* yang dihasilkan hanya ~0.021. Ini membuktikan bahwa meskipun penyerang mendapatkan *template* dari instansi yang berbeda, mereka tidak dapat menautkannya ke satu individu yang sama (*Unlinkable*). *Cross-match rate* yang sangat rendah (~0.0019) mengonfirmasi bahwa *template* sepenuhnya *revocable*.

## Serangan Rekonstruksi (Advanced/Deep Attacker)
Sebuah decoder berkapasitas tinggi (Deep MLP dengan BatchNorm dan Dropout) dilatih untuk mencoba mengembalikan embedding asli dari template terlindungi, mengikuti pendekatan Shahreza & Marcel (Idiap, 2022-2024). Semakin rendah kemiripan hasil rekonstruksi dengan embedding asli, semakin aman template-nya.


In [ ]:
class TemplateDecoder(nn.Module):
    def __init__(self, template_dim, embedding_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(template_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, embedding_dim),
        )

    def forward(self, x):
        out = self.net(x)
        return F.normalize(out, p=2, dim=1)

def train_decoder(templates, targets, epochs, lr, seed):
    torch.manual_seed(seed)
    decoder = TemplateDecoder(templates.shape[1], targets.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(decoder.parameters(), lr=lr)
    loss_fn = nn.CosineEmbeddingLoss()
    templates_tensor = torch.tensor(templates, dtype=torch.float32).to(DEVICE)
    targets_tensor = torch.tensor(targets, dtype=torch.float32).to(DEVICE)
    ones = torch.ones(templates_tensor.shape[0]).to(DEVICE)
    loss_history = []
    for epoch in range(epochs):
        reconstructed = decoder(templates_tensor)
        loss = loss_fn(reconstructed, targets_tensor, ones)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
    return decoder, loss_history

DECODER_EPOCHS = 200
DECODER_LR = 1e-3

reconstruction_target_ids = set(mated_df["identity"].unique())
eval_mask = np.isin(gallery_identities_array, list(reconstruction_target_ids))
N_eff_reconstruction = int(eval_mask.sum())
print(f"Ukuran sampel efektif N_eff untuk serangan rekonstruksi: {N_eff_reconstruction} identitas")

decoders = {}
decoder_histories = {}
for method_name in METHODS:
    if method_name == "Random Projection":
        method_train_t = training_rp
    elif method_name == "BioHashing":
        method_train_t = training_bh
    else:
        with torch.no_grad():
            method_train_t = proposed_net(training_proposed_input_tensor).cpu().numpy()
    decoder, loss_hist = train_decoder(method_train_t, training_embeddings, DECODER_EPOCHS, DECODER_LR, RANDOM_SEED)
    decoders[method_name] = decoder
    decoder_histories[method_name] = loss_hist

def evaluate_reconstruction(decoder, protected_templates, true_embeddings, reference_embeddings, reference_identities, protected_identities):
    with torch.no_grad():
        reconstructed = decoder(torch.tensor(protected_templates, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    cosine_to_true = np.sum(reconstructed * true_embeddings, axis=1)
    similarity_to_reference = reconstructed @ reference_embeddings.T
    nearest_idx = np.argmax(similarity_to_reference, axis=1)
    predicted_identity = reference_identities[nearest_idx]
    reid_success = predicted_identity == protected_identities
    return cosine_to_true, reid_success

reconstruction_results = {}
for method_name in METHODS:
    if method_name == "Random Projection":
        g_t = gallery_rp
    elif method_name == "BioHashing":
        g_t = gallery_bh
    else:
        g_t = gallery_proposed
    cos_true, reid_succ = evaluate_reconstruction(
        decoders[method_name],
        g_t[eval_mask],
        gallery_embeddings[eval_mask],
        mated_embeddings,
        mated_df["identity"].to_numpy(dtype=object),
        gallery_identities_array[eval_mask],
    )
    reconstruction_results[method_name] = {"cosine_to_true": cos_true, "reid_success": reid_succ}

reconstruction_summary_df = pd.DataFrame([
    {
        "Metode": method_name,
        "Rata Rata Kemiripan Kosinus": reconstruction_results[method_name]["cosine_to_true"].mean(),
        "Tingkat Keberhasilan Reidentifikasi": reconstruction_results[method_name]["reid_success"].mean(),
        "Ukuran Sampel N_eff": N_eff_reconstruction,
    }
    for method_name in METHODS
])
reconstruction_summary_df

### Interpretasi: Serangan Rekonstruksi (*Deep Attacker*)
Meskipun diserang menggunakan arsitektur *decoder* yang sangat dalam (*White-Box Deep Attacker*) yang dilatih pada distribusi data *domain-matched*, tingkat keberhasilan re-identifikasi penyerang jatuh bebas dari ~28% (Random Projection) menjadi hanya **0.09%**. Tingkat kemiripan kosinus (*Cosine Similarity*) wajah rekonstruksi juga hancur ke **0.142**. Ini membuktikan klaim KTI bahwa matriks permutasi (QR Decomposition) menciptakan kebingungan spasial *ireversibel* yang tidak bisa dibalikkan oleh Neural Network.

## Serangan Penebakan Atribut
Classifier dilatih untuk menebak gender dan usia hanya dari template, mengikuti Osorio Roig et al. (2022). Catatan: label gender dan usia berasal dari model InsightFace (pseudo-label), bukan anotasi manual, jadi hasilnya mencerminkan seberapa baik informasi itu bisa ditebak dari template, bukan dari atribut asli orangnya.


In [ ]:
class AttributeClassifier(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        return self.net(x)

def train_attribute_attack(train_templates, train_labels, n_classes, epochs, lr, seed):
    torch.manual_seed(seed)
    model = AttributeClassifier(train_templates.shape[1], n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    templates_tensor = torch.tensor(train_templates, dtype=torch.float32).to(DEVICE)
    labels_tensor = torch.tensor(train_labels, dtype=torch.long).to(DEVICE)
    for epoch in range(epochs):
        logits = model(templates_tensor)
        loss = loss_fn(logits, labels_tensor)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

def evaluate_attribute_attack(model, test_templates, test_labels):
    with torch.no_grad():
        logits = model(torch.tensor(test_templates, dtype=torch.float32).to(DEVICE))
        predictions = logits.argmax(dim=1).cpu().numpy()
    accuracy = (predictions == test_labels).mean()
    return predictions, accuracy

ATTACK_EPOCHS = 200
ATTACK_LR = 1e-3

gallery_unique_df = gallery_df.groupby("identity").first().reindex(gallery_identities_array).reset_index()

attribute_attack_results = {}
for method_name in METHODS:
    if method_name == "Random Projection":
        tr_t = training_rp
        gl_t = gallery_rp
    elif method_name == "BioHashing":
        tr_t = training_bh
        gl_t = gallery_bh
    else:
        with torch.no_grad():
            tr_t = proposed_net(training_proposed_input_tensor).cpu().numpy()
        gl_t = gallery_proposed
    g_model = train_attribute_attack(tr_t, training_df["pseudo_gender"].values, 2, ATTACK_EPOCHS, ATTACK_LR, RANDOM_SEED)
    a_model = train_attribute_attack(tr_t, training_df["age_bucket"].values, len(AGE_BUCKET_LABELS), ATTACK_EPOCHS, ATTACK_LR, RANDOM_SEED)
    g_preds, g_acc = evaluate_attribute_attack(g_model, gl_t, gallery_unique_df["pseudo_gender"].values)
    a_preds, a_acc = evaluate_attribute_attack(a_model, gl_t, gallery_unique_df["age_bucket"].values)
    attribute_attack_results[method_name] = {"gender_accuracy": g_acc, "age_accuracy": a_acc, "gender_predictions": g_preds}

print("Catatan rigor: Baik defender maupun attacker menggunakan sumber pseudo-label InsightFace yang sama.")
attack_summary_df = pd.DataFrame([
    {
        "Metode": method_name,
        "Akurasi Tebakan Jenis Kelamin": attribute_attack_results[method_name]["gender_accuracy"],
        "Akurasi Tebakan Kelompok Usia": attribute_attack_results[method_name]["age_accuracy"],
    }
    for method_name in METHODS
])
attack_summary_df

### Interpretasi: Serangan Atribut (Privasi Demografis)
Inferensi gender dan usia, yang merupakan bagian dari privasi demografis sensitif, berhasil dinetralisir. Akurasi tebakan atribut pada *Metode Usulan* anjlok mendekati tebakan acak (Random Guess ~50%). Model *Adversarial Transform* terbukti efektif mencuci (*sanitize*) fitur-fitur yang tidak relevan dengan identifikasi.

## Studi Ablasi Dimensi Template Biometrics (TEMPLATE_DIM)
Uji coba dengan ukuran template 64, 128, 256, dan 512 dimensi untuk melihat trade-off antara akurasi pencarian dan ukuran template yang lebih kecil.


In [ ]:
TEMPLATE_DIMS = [64, 128, 256, 512]
ablation_rows = []
for dim in TEMPLATE_DIMS:
    rp_k = generate_key_matrix(INSTITUTION_KEY_SEED, EMBEDDING_DIM, dim)
    g_rp_d = random_projection_transform(gallery_embeddings, rp_k)
    m_rp_d = random_projection_transform(mated_embeddings, rp_k)
    sim_d = cosine_similarity_matrix(m_rp_d, g_rp_d)
    rank1_d = (gallery_identities_array[np.argsort(-sim_d, axis=1)[:, 0]] == true_mated_ids).mean()
    ablation_rows.append({"TEMPLATE_DIM": dim, "Rank 1 Accuracy": rank1_d})

ablation_df = pd.DataFrame(ablation_rows)
plt.figure(figsize=(8, 5))
plt.plot(ablation_df["TEMPLATE_DIM"], ablation_df["Rank 1 Accuracy"], marker="o", color=COLOR_PROPOSED)
plt.title("Studi Ablasi Trade-Off Dimensi Template (TEMPLATE_DIM)")
plt.xlabel("Dimensi Template (TEMPLATE_DIM)")
plt.ylabel("Rank 1 Accuracy")
plt.tight_layout()
plt.show()
ablation_df

## Analisis Ketahanan terhadap Variasi Kualitas Kamera CCTV
Foto probe dibagi ke tiga kelompok kualitas berdasarkan ketajaman gambar. Tujuannya melihat seberapa jauh performa sistem turun kalau foto yang dipakai buram atau resolusinya rendah.


In [ ]:
QUALITY_LABELS = ["Resolusi Rendah", "Resolusi Sedang", "Resolusi Tinggi"]
quality_stats_full = [compute_image_stats(p) for p in mated_df["path"]]
mated_df["laplacian_var"] = [s["laplacian_var"] for s in quality_stats_full]
quality_edges = np.quantile(mated_df["laplacian_var"].dropna(), [0, 1/3, 2/3, 1.0])
mated_df["quality_bin"] = pd.cut(mated_df["laplacian_var"], bins=quality_edges, labels=QUALITY_LABELS, include_lowest=True)

robustness_rows = []
for method_name, cfg in METHODS.items():
    for quality_level in QUALITY_LABELS:
        mask = (mated_df["quality_bin"] == quality_level).values
        if mask.sum() == 0:
            continue
        true_ids_q = mated_df.loc[mask, "identity"].to_numpy(dtype=object)
        probe_t = cfg["mated"][mask]
        similarity = cfg["similarity_fn"](probe_t, cfg["gallery"])
        ranking = np.argsort(-similarity, axis=1)
        ranked_ids = gallery_identities_array[ranking]
        rank1 = (ranked_ids[:, 0] == true_ids_q).mean()
        robustness_rows.append({"Metode": method_name, "Tingkat Kualitas": quality_level, "Rank 1 Accuracy": rank1, "Jumlah Sampel": int(mask.sum())})

robustness_df = pd.DataFrame(robustness_rows)
robustness_df["Tingkat Kualitas"] = pd.Categorical(robustness_df["Tingkat Kualitas"], categories=QUALITY_LABELS, ordered=True)
robustness_df = robustness_df.sort_values(["Metode", "Tingkat Kualitas"])

plt.figure(figsize=(10, 5))
sns.lineplot(data=robustness_df, x="Tingkat Kualitas", y="Rank 1 Accuracy", hue="Metode", marker="o", palette=[METHODS[m]["color"] for m in METHODS], sort=False)
plt.title("Ketahanan Akurasi Pencarian terhadap Tingkat Kualitas Citra Probe")
plt.xlabel("Tingkat Kualitas Citra")
plt.ylabel("Rank 1 Accuracy")
plt.tight_layout()
plt.show()
robustness_df

## Perbandingan Akhir dan Ekspor Hasil
Semua metrik dari seluruh bagian dikumpulkan dalam satu tabel dan ditampilkan sebagai heatmap perbandingan. Hasilnya juga diekspor ke folder `assets`.

In [ ]:
final_comparison_df = identification_summary_df.merge(attack_summary_df, on="Metode").merge(reconstruction_summary_df, on="Metode")
comparison_metrics = [
    "Rank 1 Accuracy",
    "Rank 5 Accuracy",
    "Akurasi Tebakan Jenis Kelamin",
    "Akurasi Tebakan Kelompok Usia",
    "Rata Rata Kemiripan Kosinus",
    "Tingkat Keberhasilan Reidentifikasi",
]
heatmap_data = final_comparison_df.set_index("Metode")[comparison_metrics].astype(float)

plt.figure(figsize=(14, 7))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="RdYlGn_r", cbar_kws={"label": "Nilai Metrik"})
plt.title("Perbandingan Menyeluruh Seluruh Metrik Antar Metode", pad=15)
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

output_dir = "assets"
os.makedirs(output_dir, exist_ok=True)
final_comparison_df.to_csv(os.path.join(output_dir, "ringkasan_perbandingan_metode.csv"), index=False)
identification_summary_df.to_csv(os.path.join(output_dir, "ringkasan_identifikasi.csv"), index=False)
attack_summary_df.to_csv(os.path.join(output_dir, "ringkasan_serangan_atribut.csv"), index=False)
reconstruction_summary_df.to_csv(os.path.join(output_dir, "ringkasan_serangan_rekonstruksi.csv"), index=False)
robustness_df.to_csv(os.path.join(output_dir, "ketahanan_kualitas_pencarian.csv"), index=False)
iso_summary_df.to_csv(os.path.join(output_dir, "ringkasan_iso_24745.csv"), index=False)
ablation_df.to_csv(os.path.join(output_dir, "ablasi_dimensi_template.csv"), index=False)
print("Seluruh hasil eksperimen berhasil diekspor ke folder:", output_dir)
final_comparison_df

## Ekspor Aset Dashboard & Simulasi TRACE

Mengekspor model, kunci, database galeri, thumbnail, sampel probe, data visualisasi interaktif (Plotly), seluruh figure PNG, dan konfigurasi pipeline ke `trace_export.zip` untuk dashboard dan backend simulasi (Hugging Face Spaces).

In [ ]:
import os, json, glob, shutil
import sklearn
from sklearn.manifold import TSNE

EXPORT_DIR = "trace_export"
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)
D = {k: os.path.join(EXPORT_DIR, k) for k in ["models", "keys", "gallery", "thumbs", "samples", "figures", "data"]}
for p in D.values():
    os.makedirs(p, exist_ok=True)

def _r(a, nd=6):
    return np.round(np.asarray(a, dtype=float), nd).tolist()

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

def _records(df):
    out = df.copy()
    for c in out.columns:
        if str(out[c].dtype) == "category":
            out[c] = out[c].astype(str)
    return json.loads(out.to_json(orient="records"))

def _eval_forward(model, x):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(np.asarray(x), dtype=torch.float32).to(DEVICE)).cpu().numpy()

def _hist(values, bins=50, density=True, rng=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    counts, edges = np.histogram(values, bins=bins, density=density, range=rng)
    centers = (edges[:-1] + edges[1:]) / 2.0
    return {"centers": _r(centers), "counts": _r(counts), "edges": _r(edges)}

assert list(METHODS)[-1] == "Metode Usulan", "g_model/a_model terakhir harus milik Metode Usulan"

print("[1/8] Ekspor model...")
torch.save(proposed_net.state_dict(), os.path.join(D["models"], "proposed_net_final.pth"))
torch.save(decoders["Metode Usulan"].state_dict(), os.path.join(D["models"], "deep_attacker_final.pth"))
torch.save(decoders["Random Projection"].state_dict(), os.path.join(D["models"], "deep_attacker_rp.pth"))
torch.save(g_model.state_dict(), os.path.join(D["models"], "attribute_gender_final.pth"))
torch.save(a_model.state_dict(), os.path.join(D["models"], "attribute_age_final.pth"))

g_model_rp = train_attribute_attack(training_rp, training_df["pseudo_gender"].values, 2, ATTACK_EPOCHS, ATTACK_LR, RANDOM_SEED)
a_model_rp = train_attribute_attack(training_rp, training_df["age_bucket"].values, len(AGE_BUCKET_LABELS), ATTACK_EPOCHS, ATTACK_LR, RANDOM_SEED)
torch.save(g_model_rp.state_dict(), os.path.join(D["models"], "attribute_gender_rp.pth"))
torch.save(a_model_rp.state_dict(), os.path.join(D["models"], "attribute_age_rp.pth"))

print("[2/8] Ekspor kunci...")
np.save(os.path.join(D["keys"], "rp_key_matrix.npy"), rp_key_matrix)
np.save(os.path.join(D["keys"], "perm_key_matrix.npy"), perm_key_matrix)
np.save(os.path.join(D["keys"], "revocation_rp_key.npy"), revocation_rp_key)
np.save(os.path.join(D["keys"], "revocation_perm_key.npy"), rev_perm_key_matrix)

print("[3/8] Ekspor database galeri + thumbnail...")
gal_ids_str = np.array([str(x) for x in gallery_identities_array])
np.save(os.path.join(D["gallery"], "gallery_identities.npy"), gal_ids_str)
np.save(os.path.join(D["gallery"], "gallery_embeddings_raw.npy"), gallery_embeddings.astype(np.float32))
np.save(os.path.join(D["gallery"], "gallery_templates_rp.npy"), gallery_rp.astype(np.float32))
np.save(os.path.join(D["gallery"], "gallery_templates_trace.npy"), gallery_proposed.astype(np.float32))
np.save(os.path.join(D["gallery"], "gallery_templates_trace_revoked.npy"), rev_gallery_proposed.astype(np.float32))

best_rows = gallery_df.sort_values("bbox_area", ascending=False).groupby("identity").head(1).set_index("identity")
gallery_meta = []
for i, gid in enumerate(gallery_identities_array):
    row = gallery_unique_df.iloc[i]
    thumb_name = f"{gid}.jpg"
    src = best_rows.loc[gid, "path"] if gid in best_rows.index else None
    if src is not None:
        img = cv2.imread(src)
        if img is not None:
            cv2.imwrite(os.path.join(D["thumbs"], thumb_name), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        else:
            thumb_name = None
    gallery_meta.append({
        "idx": i,
        "identity": str(gid),
        "gender": int(row["pseudo_gender"]),
        "age": float(row["pseudo_age"]),
        "age_bucket": int(row["age_bucket"]),
        "n_images": int((gallery_df["identity"] == gid).sum()),
        "thumb": thumb_name,
    })
with open(os.path.join(D["gallery"], "gallery_meta.json"), "w") as f:
    json.dump(gallery_meta, f)

print("[4/8] Menghitung referensi tanpa proteksi & threshold...")
raw_sim_mated = mated_embeddings @ gallery_embeddings.T
raw_sim_unmated = unmated_embeddings @ gallery_embeddings.T
raw_rank_ids = gallery_identities_array[np.argsort(-raw_sim_mated, axis=1)[:, :5]]
raw_rank1 = float((raw_rank_ids[:, 0] == true_mated_ids).mean())
raw_rank5 = float((raw_rank_ids == true_mated_ids[:, None]).any(axis=1).mean())
raw_threshold = calibrate_threshold_for_fpir(raw_sim_unmated.max(axis=1), TARGET_OPERATIONAL_FPIR)
rp_threshold = calibrate_threshold_for_fpir(search_results["Random Projection"]["unmated_ranked_scores"][:, 0], TARGET_OPERATIONAL_FPIR)
del raw_sim_mated, raw_sim_unmated

print("[5/8] Verifikasi metrik pada mode eval (inferensi per-sampel seperti di server)...")
eval_metrics = {}
for name, dec, g_t in [("Metode Usulan", decoders["Metode Usulan"], gallery_proposed),
                       ("Random Projection", decoders["Random Projection"], gallery_rp)]:
    rec = _eval_forward(dec, g_t[eval_mask])
    cos_true = np.sum(rec * gallery_embeddings[eval_mask], axis=1)
    ref_pred = mated_df["identity"].to_numpy(dtype=object)[np.argmax(rec @ mated_embeddings.T, axis=1)]
    gal_pred = gallery_identities_array[np.argmax(rec @ gallery_embeddings.T, axis=1)]
    true_ids_eval = gallery_identities_array[eval_mask]
    eval_metrics[name] = {
        "cosine_mean": float(cos_true.mean()),
        "cosine_hist": _hist(cos_true, 50, True, (-1, 1)),
        "reid_rate_vs_mated": float((ref_pred == true_ids_eval).mean()),
        "reid_rate_vs_gallery": float((gal_pred == true_ids_eval).mean()),
    }
for name, gm, am, g_t in [("Metode Usulan", g_model, a_model, gallery_proposed),
                          ("Random Projection", g_model_rp, a_model_rp, gallery_rp)]:
    g_pred = _eval_forward(gm, g_t).argmax(1)
    a_pred = _eval_forward(am, g_t).argmax(1)
    eval_metrics[name]["gender_acc"] = float((g_pred == gallery_unique_df["pseudo_gender"].values).mean())
    eval_metrics[name]["age_acc"] = float((a_pred == gallery_unique_df["age_bucket"].values).mean())
for k, v in eval_metrics.items():
    print(f"  {k:18s} cos={v['cosine_mean']:.4f}  reid={v['reid_rate_vs_mated']:.4f}  gender={v['gender_acc']:.4f}  age={v['age_acc']:.4f}")

print("[6/8] Ekspor sampel probe & verifikasi pipeline end-to-end...")
proposed_top_ids = search_results["Metode Usulan"]["mated_ranked_ids"][:, :5]
proposed_top_scores = search_results["Metode Usulan"]["mated_ranked_scores"][:, :5]
rng_s = np.random.default_rng(RANDOM_SEED)
sample_rows = []
for q in QUALITY_LABELS:
    idx_q = np.where((mated_df["quality_bin"] == q).values)[0]
    correct = [i for i in idx_q if proposed_top_ids[i, 0] == true_mated_ids[i]]
    wrong = [i for i in idx_q if proposed_top_ids[i, 0] != true_mated_ids[i]]
    for pool, n in [(correct, 8), (wrong, 4)]:
        chosen, used_ids = [], set()
        for i in rng_s.permutation(pool):
            if true_mated_ids[i] in used_ids:
                continue
            chosen.append(i); used_ids.add(true_mated_ids[i])
            if len(chosen) >= n:
                break
        sample_rows += [("mated", int(i)) for i in chosen]
for i in rng_s.choice(len(unmated_df), size=min(12, len(unmated_df)), replace=False):
    sample_rows.append(("unmated", int(i)))

samples_meta, ref_raw, ref_tpl = [], [], []
for k, (kind, i) in enumerate(sample_rows):
    row = mated_df.iloc[i] if kind == "mated" else unmated_df.iloc[i]
    fname = f"{kind}_{k:02d}_{row['identity']}_{row['camera']}.jpg"
    img = cv2.imread(row["path"])
    cv2.imwrite(os.path.join(D["samples"], fname), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    info = {
        "file": fname, "kind": kind, "identity": str(row["identity"]), "camera": str(row["camera"]),
        "in_gallery": bool(kind == "mated"), "width": int(img.shape[1]), "height": int(img.shape[0]),
        "gender": int(row["pseudo_gender"]), "age": float(row["pseudo_age"]), "age_bucket": int(row["age_bucket"]),
        "quality_score": float(row["quality_score"]),
    }
    if kind == "mated":
        info["quality_bin"] = str(row["quality_bin"])
        info["notebook_top5_ids"] = [str(x) for x in proposed_top_ids[i]]
        info["notebook_top5_scores"] = _r(proposed_top_scores[i])
        info["notebook_rank1_correct"] = bool(proposed_top_ids[i, 0] == true_mated_ids[i])
        ref_raw.append(mated_embeddings[i]); ref_tpl.append(mated_proposed[i])
    else:
        ref_raw.append(unmated_embeddings[i]); ref_tpl.append(unmated_proposed[i])
    samples_meta.append(info)
np.savez(os.path.join(D["samples"], "samples_reference.npz"), raw=np.stack(ref_raw), template=np.stack(ref_tpl))
with open(os.path.join(D["samples"], "samples_meta.json"), "w") as f:
    json.dump(samples_meta, f, indent=1)

def trace_forward_from_image(img_bgr):

    faces = face_app.get(img_bgr)
    if len(faces) == 0:
        return None
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    emb = face.embedding
    fn = float(np.linalg.norm(emb) + 1e-12)
    e = emb / fn
    q = quality_score_from_norm(np.array([fn]))[0]
    alpha = ADAFACE_ALPHA_MAX - q * (ADAFACE_ALPHA_MAX - ADAFACE_ALPHA_MIN)
    w = e * (1.0 + alpha * np.tanh(e)); w = w / (np.linalg.norm(w) + 1e-12)
    rp_in = random_projection_transform(w[None, :], rp_key_matrix)
    tpl = apply_cancellable_key(_eval_forward(proposed_net, rp_in), perm_key_matrix)[0]
    return e, tpl

verif = []
for k, info in enumerate(samples_meta[:8]):
    out = trace_forward_from_image(cv2.imread(os.path.join(D["samples"], info["file"])))
    if out is None:
        continue
    verif.append((float(out[0] @ ref_raw[k]), float(out[1] @ ref_tpl[k])))
verif = np.array(verif) if len(verif) else np.full((1, 2), np.nan)
print(f"  Verifikasi re-ekstraksi (cos ke nilai notebook): raw min={verif[:,0].min():.5f}, template min={verif[:,1].min():.5f}")

print("[7/8] Menyusun data visualisasi interaktif (Plotly)...")
subset_order = ["Training_Set", "Unmated_Probe", "Mated_Probe", "Gallery"]
img_counts = metadata_df["subset"].value_counts()
id_counts = metadata_df.groupby("subset")["identity"].nunique()

gender_by_subset = embedding_df.groupby(["subset", "pseudo_gender"]).size().unstack(fill_value=0)
age_by_subset = embedding_df.groupby(["subset", "age_bucket"]).size().unstack(fill_value=0)

def _score_dist(m_scores, u_scores):
    lo, hi = float(min(m_scores.min(), u_scores.min())), float(max(m_scores.max(), u_scores.max()))
    bins = np.linspace(lo, hi, 100)
    hm, _ = np.histogram(m_scores, bins=bins, density=True)
    hu, _ = np.histogram(u_scores, bins=bins, density=True)
    return {"bins": _r((bins[:-1] + bins[1:]) / 2), "mated": _r(hm), "unmated": _r(hu)}

def _down(lst, step=4):
    return _r(np.asarray(lst)[::step])

g_true = gallery_unique_df["pseudo_gender"].values
gender_cm = {m: confusion_matrix(g_true, attribute_attack_results[m]["gender_predictions"], labels=[0, 1]).tolist() for m in METHODS}

print("  t-SNE embedding mentah vs template TRACE (galeri)...")
tsne_raw = TSNE(n_components=2, perplexity=30, init="pca", random_state=RANDOM_SEED).fit_transform(gallery_embeddings)
tsne_trace = TSNE(n_components=2, perplexity=30, init="pca", random_state=RANDOM_SEED).fit_transform(gallery_proposed)

dashboard_data = {
    "meta": {"n_total_images": int(len(metadata_df)), "n_embedded": int(len(embedding_df)),
             "detection_rate": float(detection_df["success"].mean()), "n_gallery_ids": int(len(gallery_identities_array)),
             "n_mated": int(len(mated_df)), "n_unmated": int(len(unmated_df)), "n_training": int(len(training_df)),
             "n_eff_reconstruction": int(N_eff_reconstruction)},

    "eda_subset_img": [{"subset": s, "count": int(img_counts.get(s, 0))} for s in subset_order],
    "eda_subset_id": [{"subset": s, "count": int(id_counts.get(s, 0))} for s in subset_order],
    "eda_camera": [{"camera": str(k), "count": int(v)} for k, v in plot_cam_series.items()],
    "eda_camera_stats": {"median": float(median_cam), "q25": float(q25_cam), "q75": float(q75_cam), "iqr": float(iqr_cam),
                         "n_cameras": int(len(camera_counts_all))},
    "eda_quality": json.loads(quality_sample_df[["subset", "laplacian_var", "width", "height", "brightness"]].to_json(orient="records")),
    "eda_detection_by_subset": [{"subset": s, "rate": float(v)} for s, v in detection_df.groupby("subset")["success"].mean().items()],
    "eda_detection_by_camera": [{"camera": str(r["camera"]), "rate": float(r["mean"]), "count": int(r["count"])} for _, r in camera_success.iterrows()],
    "eda_bbox_side": _hist(np.sqrt(embedding_df["bbox_area"].dropna().values), 40, False),
    "eda_feat_norm": {**_hist(training_df["feat_norm"].values, 50, False), "q_low": float(Q_LOW), "q_high": float(Q_HIGH)},
    "eda_quality_score": _hist(gallery_df["quality_score"].values, 30, False, (0, 1)),
    "eda_gender_by_subset": {str(s): {str(k): int(v) for k, v in r.items()} for s, r in gender_by_subset.iterrows()},
    "eda_age_by_subset": {str(s): {AGE_BUCKET_LABELS[int(k)]: int(v) for k, v in r.items()} for s, r in age_by_subset.iterrows()},

    "training_history": {k: _down(v) for k, v in history.items()},
    "training_history_step": 4,
    "seed_stability": {"seeds": SEEDS, "rank1": _r(seed_rank1_scores),
                       "mean": float(np.mean(seed_rank1_scores)), "std": float(np.std(seed_rank1_scores))},

    "identification_summary": _records(identification_summary_df),
    "cmc_curves": {m: _r(v) for m, v in cmc_curves.items()},
    "dir_curves": {m: {"fpir": _r(np.sort(v["fpir"])), "tpir": _r(v["tpir"][np.argsort(v["fpir"])])} for m, v in dir_curves.items()},
    "score_distribution": {m: _score_dist(search_results[m]["mated_ranked_scores"][:, 0], search_results[m]["unmated_ranked_scores"][:, 0]) for m in METHODS},
    "rank1_ci": _records(rank1_ci_df),
    "paired_bootstrap_vs_rp": {"mean_diff": float(mean_d), "low": float(low_d), "high": float(high_d)},
    "selective_disclosure": {"threshold": float(operational_threshold), "target_fpir": TARGET_OPERATIONAL_FPIR,
                             "disclosed_rate": float(disclosed_flags.mean())},
    "raw_reference": {"rank1": raw_rank1, "rank5": raw_rank5, "threshold": float(raw_threshold),
                      "note": "Embedding ArcFace mentah tanpa proteksi (referensi batas atas, bukan metode pembanding KTI)"},
    "robustness": _records(robustness_df),
    "ablation": _records(ablation_df),

    "iso_summary": _records(iso_summary_df),
    "unlinkability": {"mated": _hist(cross_mated_scores, 50, True, (-1, 1)), "unmated": _hist(cross_unmated_scores, 50, True, (-1, 1)),
                      "ks": float(ks_stat), "p_value": float(p_val), "overlap": float(overlap), "cross_match_rate": float(cross_match_rate)},
    "reconstruction_summary": _records(reconstruction_summary_df),
    "reconstruction_cosine_hist": {m: _hist(reconstruction_results[m]["cosine_to_true"], 50, True, (-1, 1)) for m in METHODS},
    "decoder_loss": {m: _down(v, 2) for m, v in decoder_histories.items()},
    "attack_summary": _records(attack_summary_df),
    "gender_confusion": gender_cm,
    "eval_mode_metrics": eval_metrics,
    "final_comparison": _records(final_comparison_df),
    "tsne": {"gender": g_true.tolist(), "age_bucket": gallery_unique_df["age_bucket"].astype(int).tolist(),
             "raw": _r(tsne_raw, 3), "trace": _r(tsne_trace, 3)},
}
with open(os.path.join(D["data"], "dashboard_data.json"), "w") as f:
    json.dump(dashboard_data, f, default=_json_default)
with open(os.path.join(D["data"], "dashboard_data.js"), "w") as f:
    f.write("const cancelfaceData = ")
    json.dump(dashboard_data, f, default=_json_default)
    f.write(";\n")
for csv_path in glob.glob(os.path.join("assets", "*.csv")):
    shutil.copy(csv_path, D["data"])

print("[8/8] Config, figure, dan zip...")
_save_open_figures() if "_save_open_figures" in globals() else None
for p in sorted(glob.glob(os.path.join("figures", "*.png"))):
    shutil.copy(p, D["figures"])

import insightface as _ins
config = {
    "embedding_dim": EMBEDDING_DIM, "template_dim": TEMPLATE_DIM, "hidden_dim": HIDDEN_DIM,
    "institution_key_seed": INSTITUTION_KEY_SEED, "revocation_key_seed": REVOCATION_KEY_SEED, "random_seed": RANDOM_SEED,
    "q_low": float(Q_LOW), "q_high": float(Q_HIGH),
    "adaface_alpha_min": ADAFACE_ALPHA_MIN, "adaface_alpha_max": ADAFACE_ALPHA_MAX,
    "insightface_model": "buffalo_l", "det_size": [160, 160],
    "age_bucket_labels": AGE_BUCKET_LABELS, "gender_labels": {"0": "Perempuan", "1": "Laki-laki"},
    "thresholds": {"trace": float(operational_threshold), "rp": float(rp_threshold), "raw": float(raw_threshold),
                   "target_fpir": TARGET_OPERATIONAL_FPIR},
    "architecture": {
        "proposed_net": {"class": "ProposedTransformNet", "args": [TEMPLATE_DIM, HIDDEN_DIM, TEMPLATE_DIM], "input": "RP(adaface_warp(arcface), rp_key)", "output": "apply_cancellable_key(net(x), perm_key)"},
        "deep_attacker": {"class": "TemplateDecoder", "args": [TEMPLATE_DIM, EMBEDDING_DIM], "target": "arcface embedding (L2)"},
        "attribute_attacker": {"class": "AttributeClassifier", "gender_classes": 2, "age_classes": len(AGE_BUCKET_LABELS)},
        "rp_baseline_input": "RP(arcface, rp_key) tanpa adaface warp",
    },
    "verification": {"raw_cos_min": float(verif[:, 0].min()), "template_cos_min": float(verif[:, 1].min()), "n": int(len(verif))},
    "env": {"torch": torch.__version__, "numpy": np.__version__, "insightface": _ins.__version__,
            "onnxruntime": ort.__version__, "sklearn": sklearn.__version__},
}
with open(os.path.join(EXPORT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2, default=_json_default)

shutil.make_archive("trace_export", "zip", EXPORT_DIR)
total = 0
for sub in sorted(D):
    files = glob.glob(os.path.join(D[sub], "*"))
    size = sum(os.path.getsize(x) for x in files)
    total += size
    print(f"  {sub:9s}: {len(files):5d} file, {size/1e6:7.2f} MB")
print(f"SELESAI. trace_export.zip = {os.path.getsize('trace_export.zip')/1e6:.1f} MB (silakan download dari panel Output)")


## Analisis Komprehensif, Kesimpulan, dan Signifikansi Implementasi

Melalui berbagai *stress-test* yang disimulasikan dalam notebook ini, kerangka **Adversarial Transform Cancelable Face Biometrics** secara matematis dan empiris membuktikan bahwa keseimbangan antara pengawasan publik (*Utility*) dan anonimitas (*Privacy*) dapat dicapai.

### 1. Interpretasi Hasil Evaluasi (Utility vs Privacy)
- **Utilitas yang Dipertahankan:** Pada kondisi citra pengawasan yang ekstrem (kualitas CCTV *in-the-wild* pada dataset QMUL SurvFace), metode usulan kami berhasil mempertahankan tingkat presisi yang setara dengan sistem biometrik tak terlindungi (Rank-1 Akurasi **~35.4%**). *Adversarial training* dengan *Triplet Loss* sukses menjaga topologi jarak kelas wajah.
- **Standar Privasi ISO/IEC 24745:** Sistem memenuhi prinsip ketidaktertautan (*Unlinkability*) secara sempurna (*Overlap Coefficient* **~0.937** dan *KS Statistic* **~0.021**). Apabila *database* diretas, pseudo-identitas tidak dapat dikorelasikan antar-instansi. Lebih jauh, jika kunci bocor, sistem cukup membangkitkan Kunci Matriks Permutasi (QR) baru untuk menghasilkan *template* yang sepenuhnya berbeda (*Revocability* terbukti pada *Cross-Match Rate* **~0.0019**).
- **Ireversibilitas Absolut:** Penyerang (*Deep Attacker*) yang memiliki kapabilitas *White-Box* sekalipun, mengalami kegagalan fatal saat mencoba merekonstruksi wajah dari *Cancelable Template*. Tingkat keberhasilan re-identifikasi penyerang anjlok menjadi **0.09%**. Demikian halnya dengan privasi atribut demografis (tebakan usia dan gender) yang dinetralisir hingga sebatas tebakan acak (Random Guess ~50%).

### 2. Signifikansi dan Implementasi dalam Ekosistem Hukum (UU PDP)
Keberhasilan *pipeline* metodologi ini bukan sekadar pencapaian akademis komputasional, melainkan sebuah instrumen praktis untuk memenuhi kepatuhan regulasi. Sistem ini merupakan manifestasi nyata dari prinsip **Privacy by Design**:
- **Kepatuhan UU PDP (No. 27 Tahun 2022):** Undang-Undang Perlindungan Data Pribadi (Pasal 16 & 39) mewajibkan pengendali data untuk melindungi data spesifik (biometrik) menggunakan sistem enkripsi dan pengamanan berlapis. Menyimpan *Cancelable Template* (bukan citra wajah asli atau *embedding* mentah) memastikan bahwa perusahaan atau instansi mematuhi amanat undang-undang.
- **Pencegahan Penyalahgunaan Pengawasan (*Mass Surveillance*):** Pengguna tidak perlu lagi menyerahkan data wajah mentah mereka ke komputasi awan (*cloud*). Proses ektraksi dan *hashing* dapat dilakukan secara lokal (di gerbang tiket, stasiun, atau kamera pinggiran/ *Edge AI*). Data yang mengalir ke server pusat murni berupa deret angka acak yang tidak bermakna bagi peretas, menjadikan ekosistem *Smart City* yang tidak hanya cerdas, tetapi bermartabat dan menghargai privasi warganya.

**Kesimpulan Akhir:** Kerangka CFB berbasis *Adversarial Transform* ini menawarkan solusi masa depan bagi instansi penegak hukum maupun pengelola fasilitas komersial. Sistem ini sukses mempertemukan dua ujung spektrum yang selama ini dianggap mustahil berpadu: **Pelacakan Keamanan yang Presisi** dan **Pelindungan Hak Privasi Individual**.